# **Notebook 02 - Data Cleaning**

**Output:**
- `data/interim/transactions_clean.parquet` - sales transactions only, all customers (for product/revenue analysis)
- `data/interim/transactions_customer_level.parquet` - sales transactions with a valid `CustomerID` only (for customer modeling: RFM, CLV, churn)
- `data/interim/cancellations.parquet` - separated cancellation records (useful as a churn signal feature later)

**Cleaning Rule** (derived from Notebook 01 findings):
1. Separate cancellation (Invoice starts with `C`) into their own table
2. Drop non-product StockCode (POST, BANK CHARGES, M, ect.)
3. Drop zero/negative prices and quantities (for the sales table)
4. Investigate and decide on extreme outliers (Quantity > 10,000)
5. Strip whitespace and normalize `Description`
6. Create derived columns: `Revenue`, `Year`, `Month`, `DayOfWeek`, `Hour`
7. Save as parquet (faster reload, preserves dtypes)

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_PATH = Path('../data/raw/online_retail_II.csv')
INTERIM_DIR = Path('../data/interim/')
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load raw data

In [2]:
df = pd.read_csv(
    RAW_PATH,
    dtype={'Invoice': str, 'StockCode': str, 'Customer ID': str},
    parse_dates=['InvoiceDate']
)
df = df.rename(columns={'Customer ID': 'CustomerID'})

print(f'Raw shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

# Track how many rows we drop at each step
audit = [('raw_data', len(df))]

Raw shape: 1,067,371 rows x 8 columns


## 2. Separate cancellations

Cancellations (Invoice starts with `C`) are real-business events - we don't throw them away. We pull them into their own table so we can:
- Compute return-rate per customer as a feature later.
- Optionally net them against sales for revenue calculations.

But they shouldn't be mixed into the sales table because their negative quantities will skew aggregations.

In [3]:
is_cancellation = df['Invoice'].str.startswith('C')

cancellations = df[is_cancellation].copy()
df = df[~is_cancellation].copy()

print(f'Cancellations separated: {len(cancellations):,} rows')
print(f'Sales remaining: {len(df):,} rows')
audit.append(('after_separating_cancellations', len(df)))

Cancellations separated: 19,494 rows
Sales remaining: 1,047,877 rows


## 3. Drop non-product StockCode

From Notebook 01, we identified admin/fee codes that aren't real products. We define an explicit list rather than using a heuristic - explicit is safer and reviewable

**Common non-product codes in this dateset:**
- `POST`, `DOT` - postage/shipping charges
- `M` - manual adjustments
- `BANK CHARGES`, `AMAZONFEE` - Fees
- `D` - discount 
- `PADS` - pads to match all sales
- `CRUK` - charity donation
- `B` - adjust bad debt
- `TEST001`, `TEST002`, `gift_0001` - test/gift card placeholders

In [4]:
for code in ['S', 'ADJUST']:
    rows = df[df['StockCode'] == code]
    print(f"\n=== {code} ({len(rows)} rows) ===")
    print(f"Description: {rows['Description'].value_counts().head(5).to_dict()}")
    print(f"Price range: {rows['Price'].min()} to {rows['Price'].max()}")
    print(f"Quantity range: {rows['Quantity'].min()} to {rows['Quantity'].max()}")


=== S (3 rows) ===
Description: {'SAMPLES': 3}
Price range: 30.0 to 73.8
Quantity range: 1 to 1

=== ADJUST (36 rows) ===
Description: {'Adjustment by john on 26/01/2010 16': 20, 'Adjustment by john on 26/01/2010 17': 16}
Price range: 4.57 to 5117.03
Quantity range: 1 to 1


In [5]:
NON_PRODUCT_CODES = ['POST', 'DOT', 'M', 'BANK CHARGES', 'D', 'C2', 'PADS', 'AMAZONFEE', 'CRUK', 'B', 'TEST001', 'TEST002', 'S', 'ADJUST']

before = len(df)
df = df[~df['StockCode'].isin(NON_PRODUCT_CODES)].copy()

# Also drop any StockCode starting with 'gift_' (gift card placeholders)
df = df[~df['StockCode'].str.startswith('gift_')].copy()

dropped = before - len(df)
print(f'Dropped {dropped:,} rows of non-product StockCodes')
print(f'Remaining: {len(df):,} rows')
audit.append(('after_dropping_non_products', len(df)))

Dropped 4,713 rows of non-product StockCodes
Remaining: 1,043,164 rows


## 4. Drop zero and negative quantities/prices

After removing cancellations, any remaining negatives are data errors. Zero quantities and zero prices are also bad data for a sales table.

**Why not impute?** Imputing prices for products would invent revenue that didn't exist. Better to drop and be honest about data loss

In [6]:
print('Before filtering:')
print(f'    Quantity <= 0: {(df["Quantity"] <= 0).sum():,}')
print(f'    Price <= 0: {(df["Price"] <= 0).sum():,}')

before = len(df)
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)].copy()
dropped = before - len(df)
print(f'\nDropped {dropped:,} rows with non-positive Quantity or Price')
print(f'Remaining: {len(df):,} rows')
audit.append(('after_dropping_nonpositive', len(df)))

Before filtering:
    Quantity <= 0: 3,457
    Price <= 0: 6,152

Dropped 6,152 rows with non-positive Quantity or Price
Remaining: 1,037,012 rows


## 5. Inspect extreme outliers

Notebook 01 showed some Quantity values in the 80k range. We need to decide: are these legitimate huge B2B orders, or data errors? We look before we decide

In [7]:
extreme = df[df['Quantity'] > 5000].sort_values('Quantity', ascending=False)
print(f'Rows with Quantity > 5000: {len(extreme):,}')
extreme[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price', 'CustomerID', 'Country']].head(20)

Rows with Quantity > 5000: 30


,Invoice,StockCode,Description,Quantity,Price,CustomerID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2.08,16446.0,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346.0,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,0.10,13902.0,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,0.10,13902.0,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,0.10,13902.0,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,0.10,13902.0,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,0.10,13902.0,Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,0.25,17940.0,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,0.25,17940.0,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,0.25,17940.0,United Kingdom


**Decision rationale**
- If the row has a low price and round-number quantity (e.g., 80,000 units at £0.42), it's plausibly a real bulk wholesale order - keep it
- If the price is implausible low (£0.00 we already dropped) or the description is admin-like, drop it.

Since these are rare and the model will be robust to them with proper scaling (or we can log-transform), we'll **keep them but flag.** We add an `is_bulk_order` boolean for later analysis. If a few cause problems in modeling, we can revisit.

In [8]:
df['is_bulk_order'] = df['Quantity'] >= 1000
print(f'Flagged as bulk orders (>= 1000 units): {df["is_bulk_order"].sum():,} rows')

Flagged as bulk orders (>= 1000 units): 331 rows


## 6. Normalize text columns

Description have inconsistent whitespace and casing. We strip and standardize so the same product doesn't appear as multiple entries

In [9]:
df['Description'] = df['Description'].str.strip().str.upper()
df['Country'] = df['Country'].str.strip()

# Check: do any StockCode now have multiple descriptions? (Bad data: same SKU described differently)
desc_per_code = df.groupby('StockCode')['Description'].nunique()
multi_desc = desc_per_code[desc_per_code > 1]
print(f'StockCodes with multiple descriptions: {len(multi_desc):,}')

if len(multi_desc) > 0:
    print('\nExample (top 5 StockCodes with the most description variants):')
    for code in multi_desc.sort_values(ascending=False).head(5).index:
        descs = df[df['StockCode'] == code]['Description'].value_counts()
        print(f'\n  StockCode {code}:')
        for d, n in descs.items():
            print(f'    {n:>5} x "{d}')

StockCodes with multiple descriptions: 621

Example (top 5 StockCodes with the most description variants):

  StockCode 22344:
       91 x "PARTY PIZZA DISH PINK POLKADOT
       24 x "PARTY PIZZA DISH PINK RETROSPOT
       19 x "PARTY PIZZA DISH PINK WHITE SPOT
        2 x "PARTY PIZZA DISH PINK+WHITE SPOT

  StockCode 22345:
       87 x "PARTY PIZZA DISH BLUE POLKADOT
       28 x "PARTY PIZZA DISH BLUE RETROSPOT
       17 x "PARTY PIZZA DISH BLUE WHITE SPOT
        2 x "PARTY PIZZA DISH BLUE+WHITE SPOT

  StockCode 22346:
       72 x "PARTY PIZZA DISH GREEN POLKADOT
       19 x "PARTY PIZZA DISH GREEN RETROSPOT
       15 x "PARTY PIZZA DISH GREEN WHITE SPOT
        1 x "PARTY PIZZA DISH GREEN+WHITE SPOT

  StockCode 22384:
     1337 x "LUNCH BAG PINK POLKADOT
      839 x "LUNCH BAG PINK RETROSPOT
       14 x "LUNCHBAG PINK RETROSPOT
        1 x "LUNCH BAG PINK POLKADOTS

  StockCode 23196:
      281 x "VINTAGE LEAF MAGNETIC NOTEPAD
       22 x "RETRO LEAVES MAGNETIC NOTEPAD
        3 

**Decision:** For each StockCode with conflicting descriptions, we pick the most common one and  overwrite. This canonicalizes the product catalog.

In [10]:
# Build a canonical Description oer StockCode = the mode (most frequent)
canonical_desc = (
    df.dropna(subset=['Description'])
        .groupby('StockCode')['Description']
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
)
df['Description'] = df['StockCode'].map(canonical_desc)

# Sanity check
still_multi = df.groupby('StockCode')['Description'].nunique()
print(f'StockCodes with multiple descriptions after canonicalization: {(still_multi > 1).sum()}')

StockCodes with multiple descriptions after canonicalization: 0


## 7. Drop rows with null Description after canonicalization

If a StockCode has *only* null descriptios across the entire dataset, the canonicalization above leave it null. Those rows are likely junk

In [11]:
before = len(df)
df = df.dropna(subset=['Description']).copy()
dropped = before - len(df)
print(f'Dropped {dropped:,} rows with no recoverable Description')
audit.append(('after_dropping_null_desc', len(df)))

Dropped 0 rows with no recoverable Description


## 8. Derive useful columns

Add columns that every downstream notebook will need: `Revenue`, data parts. Computing these once now saves repetition later

In [12]:
df['Revenue'] = df['Quantity'] * df['Price']
df['Year'] = df['InvoiceDate'].dt.year.astype('int16')
df['Month'] = df['InvoiceDate'].dt.month.astype('int8')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour.astype('int8')
df['Date'] = df['InvoiceDate'].dt.date

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_bulk_order,Revenue,Year,Month,DayOfWeek,Hour,Date
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.40,2009,12,Tuesday,7,2009-12-01
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.00,2009,12,Tuesday,7,2009-12-01
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.00,2009,12,Tuesday,7,2009-12-01
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.80,2009,12,Tuesday,7,2009-12-01
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.00,2009,12,Tuesday,7,2009-12-01
